In [1]:
import peanuts
from azureml.core import Workspace
import pandas as pd
from peanuts.AML.orion import *

In [2]:
# Initialize Azure ML workspace and retrieve functional account password from Key Vault
ws = Workspace.from_config()
functional_account_id = 'MOSSASWRKFORCE10'
kv = ws.get_default_keyvault()
secret_name = f"{functional_account_id}-pw"
password = kv.get_secret(secret_name)

orion = Orion(functional_account=functional_account_id, password=password)

## Fetch customer comments

In [11]:
sql_query = '''
SELECT
    R.SURVEY_ID,
    R.SURVEY_RESPNS_ID,
    R.SURVEY_AA_CUST_UNIQUE_ID,
    R.SEG_DEP_DT,
    R.SEG_DEP_AIRPRT_IATA_CD,
    R.SEG_ARVL_AIRPRT_IATA_CD,
    R.OPERAT_FLIGHT_NBR,
    R.FLEET_CD,
    R.SUBFLEET_CD,
    R.CABIN_FLOWN_CD,
    T.LIKELIHOOD_RECOMMEND_ORIG_LANG_TXT,
    R.LIKELIHOOD_RECOMMEND_SCL,
    R.NET_PROMO_CATG
FROM PROD_CUST_RESRCH_SURVEY_VW.CSS_ALL_SCORES R
JOIN PROD_CUST_RESRCH_SURVEY_PII_VW.CSS_TXT_CMNT_RESPNS T
    ON T.SURVEY_ID = R.SURVEY_ID
   AND T.SURVEY_RESPNS_ID = R.SURVEY_RESPNS_ID
   AND T.SURVEY_AA_CUST_UNIQUE_ID = R.SURVEY_AA_CUST_UNIQUE_ID
WHERE LIKELIHOOD_RECOMMEND_ORIG_LANG_TXT is not null AND
	R.SEG_DEP_DT >= '2026-01-01'; 
    '''

In [12]:
df_comments_raw = orion.mq(sql_query)

pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.


In [13]:
df_comments_raw.shape

(1430569, 13)

In [14]:
df_comments_raw.head()

,SURVEY_ID,SURVEY_RESPNS_ID,SURVEY_AA_CUST_UNIQUE_ID,SEG_DEP_DT,SEG_DEP_AIRPRT_IATA_CD,SEG_ARVL_AIRPRT_IATA_CD,OPERAT_FLIGHT_NBR,FLEET_CD,SUBFLEET_CD,CABIN_FLOWN_CD,LIKELIHOOD_RECOMMEND_ORIG_LANG_TXT,LIKELIHOOD_RECOMMEND_SCL,NET_PROMO_CATG
0,SV_4SJm25NwBp7FfRs,R_7fcyCUiJviEHlLM,8.178807e+09,2026-06-28,CMH,CLT,3206,737,38R,C,One of the Flight attendants wasn't very perso...,5,Detractor
1,SV_4SJm25NwBp7FfRs,R_6JLRUWNXP3P5N2A,5.219010e+08,2026-06-08,DFW,JAX,1975,737,38K,Y,very pleasant no delays this time.,10,Promoter
2,SV_4SJm25NwBp7FfRs,R_1duU6NRx4XCTQou,8.138669e+09,2026-08-12,SLC,ORD,2883,737,38K,Y,It was late to Syracuse then leaving was even ...,3,Detractor
3,SV_4SJm25NwBp7FfRs,R_7LYGlK6I6bm2qzM,4.571319e+08,2026-01-20,DCA,CLE,5661,CRJ,CR7,Y,We were delayed 3 hours because of system issu...,2,Detractor
4,SV_4SJm25NwBp7FfRs,R_1N8FShFDPTH4tkk,1.428070e+09,2026-01-13,PHL,AUS,2691,320,A05,Y,Once on the plane I give the pilot and crew a ...,9,Promoter


In [15]:
d1 = pd.read_csv('/home/azureuser/cloudfiles/code/Users/813076/git/dark-wifi/nps_sentiment.csv')

In [16]:
d1.shape

(2980, 16)

In [17]:
d1.head()

,SURVEY_ID,SURVEY_RESPNS_ID,SURVEY_AA_CUST_UNIQUE_ID,SURVEY_LYLTY_ACCT_ID,SURVEY_LYLTY_LEVEL_CD,SEG_DEP_DT,SEG_DEP_AIRPRT_IATA_CD,SEG_ARVL_AIRPRT_IATA_CD,OPERAT_FLIGHT_NBR,FLEET_CD,SUBFLEET_CD,CABIN_FLOWN_CD,LIKELIHOOD_RECOMMEND_ORIG_LANG_TXT,LIKELIHOOD_RECOMMEND_SCL,NET_PROMO_CATG,Wifif Category
0,SV_4SJm25NwBp7FfRs,R_6LMbz6VCSrKIgWm,8054029245,TV19J66,R,8/21/2026,OKC,DFW,6203,CRJ,OO7,Y,"No delays, military preboarding and in-flight ...",9,Promoter,Satisfied with Wi-Fi Experience
1,SV_4SJm25NwBp7FfRs,R_34u5JrS528gqgXU,51071687,7152YW2,R,8/21/2026,PVR,DFW,2639,737,38K,C,"On time, and I loved the free wifi",10,Promoter,Satisfied with Wi-Fi Experience
2,SV_4SJm25NwBp7FfRs,R_3gioLrtbW8VlI6f,519743273,NaN,N,8/21/2026,BOS,LGA,4639,EMJ,E75,Y,The flight from LGA to CAE had no wifi,5,Detractor,Wi-Fi Unavailable
3,SV_4SJm25NwBp7FfRs,R_6eRh2k2FSY5nYjL,58652183,NaN,N,8/21/2026,CRP,DFW,3418,EMJ,E7M,C,No wifi,5,Detractor,Wi-Fi Unavailable
4,SV_4SJm25NwBp7FfRs,R_6mlqKAbxJI76jWq,333750122,3X14EM6,R,8/21/2026,LHR,PHX,195,777,772,C,"It was a great trip-staff excellent. However, ...",8,Passive,Wi-Fi Unavailable


In [ ]:
df_comments_raw['mention wifi'] = ''

In [10]:
len(df_comments_raw[~df_comments_raw['WIFI_SATISFACTION_SCR'].isnull()])

1396380

In [22]:
df_comments_raw.head()

,SURVEY_ID,SURVEY_RESPNS_ID,SURVEY_AA_CUST_UNIQUE_ID,SEG_DEP_DT,SEG_DEP_AIRPRT_IATA_CD,SEG_ARVL_AIRPRT_IATA_CD,OPERAT_FLIGHT_NBR,FLEET_CD,SUBFLEET_CD,CABIN_FLOWN_CD,LIKELIHOOD_RECOMMEND_ORIG_LANG_TXT,LIKELIHOOD_RECOMMEND_SCL,NET_PROMO_CATG
0,SV_4SJm25NwBp7FfRs,R_7fcyCUiJviEHlLM,8.178807e+09,2026-06-28,CMH,CLT,3206,737,38R,C,One of the Flight attendants wasn't very perso...,5,Detractor
1,SV_4SJm25NwBp7FfRs,R_6JLRUWNXP3P5N2A,5.219010e+08,2026-06-08,DFW,JAX,1975,737,38K,Y,very pleasant no delays this time.,10,Promoter
2,SV_4SJm25NwBp7FfRs,R_1duU6NRx4XCTQou,8.138669e+09,2026-08-12,SLC,ORD,2883,737,38K,Y,It was late to Syracuse then leaving was even ...,3,Detractor
3,SV_4SJm25NwBp7FfRs,R_7LYGlK6I6bm2qzM,4.571319e+08,2026-01-20,DCA,CLE,5661,CRJ,CR7,Y,We were delayed 3 hours because of system issu...,2,Detractor
4,SV_4SJm25NwBp7FfRs,R_1N8FShFDPTH4tkk,1.428070e+09,2026-01-13,PHL,AUS,2691,320,A05,Y,Once on the plane I give the pilot and crew a ...,9,Promoter


In [18]:
# Keys to merge on
merge_keys = [
    "SURVEY_ID",
    "SURVEY_RESPNS_ID",
    "SURVEY_AA_CUST_UNIQUE_ID"
]

# Keep only required columns from d1
d1_merge = (
    d1[merge_keys + ["Wifif Category"]]
    .drop_duplicates(subset=merge_keys)
    .copy()
)

# Flag rows that exist in d1
d1_merge["wifi_comment"] = 1

# Left join to retain all rows from df_comments_raw
df_final = df_comments_raw.merge(
    d1_merge,
    on=merge_keys,
    how="left"
)

# Rows not found in d1 get 0
df_final["wifi_comment"] = (
    df_final["wifi_comment"]
    .fillna(0)
    .astype(int)
)


In [19]:
print("Rows in final:", len(df_final))
print("Rows in source:", len(df_comments_raw))
print("Rows flagged as wifi comments:", df_final["wifi_comment"].sum())

Rows in final: 1430569
Rows in source: 1430569
Rows flagged as wifi comments: 874


In [21]:
df_final.to_csv('/home/azureuser/cloudfiles/code/Users/813076/git/dark-wifi/nps_comment_all.csv', index = False)

In [ ]:
csv_path = "/home/azureuser/cloudfiles/code/Users/813076/git/dark-wifi/nps_text.csv"
encodings_to_try = ["utf-8", "utf-8-sig", "cp1252", "latin1"]

last_error = None
for enc in encodings_to_try:
    try:
        data = pd.read_csv(csv_path, encoding=enc)
        print(f"Loaded CSV using encoding: {enc}")
        break
    except UnicodeDecodeError as err:
        last_error = err
else:
    raise UnicodeDecodeError(
        last_error.encoding,
        last_error.object,
        last_error.start,
        last_error.end,
        f"Could not decode {csv_path} with encodings {encodings_to_try}: {last_error.reason}",
    )

data.head()

In [ ]:
LLM_Wrapper_ENDPOINT="https://aigateway.np.aa.com/ai-cx-cntctcntr-advocate/openai/deployments/gpt-4o/chat/completions?api-version=2025-03-01-preview"
AZURE_OPENAI_KEY='d5a4dcef95aa4bd7bbe7b8327cc99821'

In [ ]:
import requests


def sentiment_analysis(prompt):
    messages = [
        {
            "role": "system",
            "content": f""" You are labeling airline survey comments about onboard WiFi.

                Task:
                Read the comment and return exactly one label from this list:
                No wifi (Wifi not working)
                No free wifi (was available, but expensive, need more free wifi)
                Issues Connecting to WiFi / poor connection
                Happy with wifi
                
                Definitions:
                No wifi (Wifi not working): WiFi unavailable, missing, not offered, no WiFi on flight.
                No free wifi (was available, but expensive, need more free wifi): customer says WiFi costs too much, should be free, had to pay, or wants more free access.
                Issues Connecting to WiFi / poor connection: WiFi exists but connection is unstable, slow, intermittent, login/connectivity failed, poor quality.
                Happy with wifi: clearly positive feedback about WiFi (liked, loved, good, great, worked well, free WiFi praised).
                
                Decision rules:
                Only read the context related to wifi/ internet. Ignore the part where customer is talking about something else.
                Prefer the most specific complaint.
                If both “no wifi” and “connection issue” appear, choose the primary complaint in tone.
                If both complaint and praise appear for the wifi, choose complaint.
                If WiFi is only mentioned neutrally with no clear sentiment, choose the closest issue if any complaint exists; otherwise choose happy with wifi only when clearly positive.
                Output label only, no explanation."""
        },
        {
            "role": "user",
            "content": f""" "{prompt}"
    """
        }
    ]
    

    payload = {"messages": messages}
    headers = {"Content-Type": "application/json", "api-key": AZURE_OPENAI_KEY}
    resp = requests.post(LLM_Wrapper_ENDPOINT, json=payload, headers=headers, timeout=30)

    return resp.json()["choices"][0]["message"]["content"]


In [ ]:
sentiment_analysis('bad wifi')

In [ ]:
import time

OUTPUT_PATH = "/home/azureuser/cloudfiles/code/Users/813076/git/dark-wifi/nps_sentiment.csv"
TEXT_COL = "LIKELIHOOD_RECOMMEND_ORIG_LANG_TXT"
SENTIMENT_COL = "Wifif Category"

ALLOWED_LABELS = {
    "No wifi (Wifi not working)": "No wifi (Wifi not working)",
    "No free wifi (was available, but expensive, need more free wifi)": "No free wifi (was available, but expensive, need more free wifi)",
    "Issues Connecting to WiFi / poor connection": "Issues Connecting to WiFi / poor connection",
    "Happy with wifi": "Happy with wifi",
}

def normalize_label(raw_label):
    if raw_label is None:
        return ""
    label = str(raw_label).strip()

    if label in ALLOWED_LABELS:
        return ALLOWED_LABELS[label]

    ll = label.lower()
    if "happy" in ll and "wifi" in ll:
        return "Happy with wifi"
    if "no free" in ll and "wifi" in ll:
        return "No free wifi (was available, but expensive, need more free wifi)"
    if ("connect" in ll or "connection" in ll or "poor" in ll or "issue" in ll) and "wifi" in ll:
        return "Issues Connecting to WiFi / poor connection"
    if "no wifi" in ll or ("wifi" in ll and "not working" in ll):
        return "No wifi (Wifi not working)"

    return ""

if SENTIMENT_COL not in data.columns:
    data['raw'] = ''
    data[SENTIMENT_COL] = ""

total_rows = len(data)
updated = 0
skipped = 0
errors = 0

for idx, row in data.iterrows():
    text = str(row.get(TEXT_COL, "") or "").strip()
    current_label = str(row.get(SENTIMENT_COL, "") or "").strip()

    if not text:
        skipped += 1
        continue
    if current_label:
        skipped += 1
        continue

    max_retries = 3
    label = ""
    for attempt in range(1, max_retries + 1):
        try:
            raw = sentiment_analysis(text)
            label = normalize_label(raw)
            break
        except Exception as exc:
            if attempt == max_retries:
                print(f"[ERROR] Row {idx}: {exc}")
            else:
                time.sleep(1.5 * attempt)

    if label:
        data.at[idx, 'raw'] = raw
        data.at[idx, SENTIMENT_COL] = label
        updated += 1
    else:
        errors += 1

    processed = idx + 1
    if processed % 50 == 0:
        data.to_csv(OUTPUT_PATH, index=False)
        print(f"Checkpoint row {processed}/{total_rows} | updated={updated} | errors={errors} | skipped={skipped}")

data.to_csv(OUTPUT_PATH, index=False)

print("Done")
print(f"Total rows: {total_rows}")
print(f"Updated labels: {updated}")
print(f"Skipped rows: {skipped}")
print(f"Rows with unresolved labels/errors: {errors}")
print(f"Saved file: {OUTPUT_PATH}")

In [ ]:
data.to_csv('/home/azureuser/cloudfiles/code/Users/813076/git/dark-wifi/nps_sentiment.csv', index=False)

In [ ]:
data['Wifif Category'].value_counts()

In [ ]:
data[['LIKELIHOOD_RECOMMEND_ORIG_LANG_TXT','Wifif Category']]

In [ ]:
# data['NET_PROMO_CATG']
pd.DataFrame(data[['Wifif Category','NET_PROMO_CATG']].value_counts()).sort_values(by = 'Wifif Category')

In [ ]:
data[(data['Wifif Category'] == 'No wifi (Wifi not working)') &(data['NET_PROMO_CATG'] == 'Promoter')]

In [ ]:
data['LIKELIHOOD_RECOMMEND_SCL']

# Fetch Wifi satisfaction scores data

sql_query = """
SELECT
    SURVEY_ID,
    SURVEY_RESPNS_ID,
    SURVEY_AA_CUST_UNIQUE_ID,
    SURVEY_LYLTY_ACCT_ID,
    SURVEY_LYLTY_LEVEL_CD,
    SEG_DEP_DT,
    SEG_DEP_AIRPRT_IATA_CD,
    SEG_ARVL_AIRPRT_IATA_CD,
    OPERAT_FLIGHT_NBR,
    FLEET_CD,
    SUBFLEET_CD,
    CABIN_FLOWN_CD,
	WIFI_SATISFACTION_SCR, 
    WIFI_SPEED_SCR, 
    WIFI_PRICE_SCR, 
    WIFI_RLBLTY_SCR, 
    WIFI_EASE_USE_SCR, 
    WIFI_CNCT_EFFRT_SCR,
    LIKELIHOOD_RECOMMEND_SCL,
    NET_PROMO_CATG
FROM PROD_CUST_RESRCH_SURVEY_VW.CSS_ALL_SCORES
WHERE SEG_DEP_DT >= DATE '2026-01-01'
"""

In [ ]:
df_raw = orion.mq(sql_query)
df_raw.head()

In [ ]:
df_raw.to_csv('/home/azureuser/cloudfiles/code/Users/813076/git/dark-wifi/nps_wifi_score_all.csv', index = False)

In [2]:
import pandas as pd

In [ ]:
d1 = pd.read_csv('/home/azureuser/cloudfiles/code/Users/813076/git/dark-wifi/nps_wifi_score_all.csv')
d1.shape

TypeError: 'tuple' object is not callable

In [4]:
d1.shape

(8417713, 20)

In [7]:
d3 = pd.read_csv('/home/azureuser/cloudfiles/code/Users/813076/git/dark-wifi/nps_wifi_score.csv')
d3.shape

(3216119, 21)

In [8]:
d2 = pd.read_csv('/home/azureuser/cloudfiles/code/Users/813076/git/dark-wifi/nps_wifi_score_all2026.csv')
d2.shape

(2050207, 20)

In [11]:
d2.head()

,SURVEY_ID,SURVEY_RESPNS_ID,SURVEY_AA_CUST_UNIQUE_ID,SURVEY_LYLTY_ACCT_ID,SURVEY_LYLTY_LEVEL_CD,SEG_DEP_DT,SEG_DEP_AIRPRT_IATA_CD,SEG_ARVL_AIRPRT_IATA_CD,OPERAT_FLIGHT_NBR,FLEET_CD,SUBFLEET_CD,CABIN_FLOWN_CD,WIFI_SATISFACTION_SCR,WIFI_SPEED_SCR,WIFI_PRICE_SCR,WIFI_RLBLTY_SCR,WIFI_EASE_USE_SCR,WIFI_CNCT_EFFRT_SCR,LIKELIHOOD_RECOMMEND_SCL,NET_PROMO_CATG
0,SV_4SJm25NwBp7FfRs,R_6dfccoDUYaEtdct,1.069030e+09,1DW91R6,R,2026-08-02,GPT,DFW,3539,EMJ,E7M,C,NaN,NaN,NaN,NaN,NaN,NaN,10,Promoter
1,SV_4SJm25NwBp7FfRs,R_3qUoFDYOCoZJbfi,4.297665e+07,W254E64,R,2026-05-22,PHL,ORD,3042,321,32C,C,100.0,NaN,NaN,NaN,NaN,NaN,6,Detractor
2,SV_4SJm25NwBp7FfRs,R_6qmHsZyymQPTjKX,8.002585e+09,NaN,N,2026-05-18,ATH,PHL,759,787,788,Y,NaN,NaN,NaN,NaN,NaN,NaN,8,Passive
3,SV_4SJm25NwBp7FfRs,R_6WVCNzZyFf690Rn,8.000967e+09,NaN,N,2026-02-20,ORD,DFW,680,737,38K,Y,50.5,NaN,NaN,NaN,NaN,NaN,1,Detractor
4,SV_4SJm25NwBp7FfRs,R_1eQOLrQ5rjQLQJP,7.786568e+06,D7R2144,T,2026-05-11,SGU,DFW,4818,CRJ,CR9,C,1.0,NaN,NaN,NaN,NaN,NaN,8,Passive


'2026-01-01'

In [10]:
d3.head()

,Unnamed: 0,SURVEY_ID,SURVEY_RESPNS_ID,SURVEY_AA_CUST_UNIQUE_ID,SURVEY_LYLTY_ACCT_ID,SURVEY_LYLTY_LEVEL_CD,SEG_DEP_DT,SEG_DEP_AIRPRT_IATA_CD,SEG_ARVL_AIRPRT_IATA_CD,OPERAT_FLIGHT_NBR,...,SUBFLEET_CD,CABIN_FLOWN_CD,WIFI_SATISFACTION_SCR,WIFI_SPEED_SCR,WIFI_PRICE_SCR,WIFI_RLBLTY_SCR,WIFI_EASE_USE_SCR,WIFI_CNCT_EFFRT_SCR,LIKELIHOOD_RECOMMEND_SCL,NET_PROMO_CATG
0,0,SV_4SJm25NwBp7FfRs,R_6WVhjKCl3YU9P48,3.519923e+06,534KX10,R,2026-03-27,GUA,MIA,1258,...,38R,C,75.25,NaN,NaN,NaN,NaN,NaN,10,Promoter
1,1,SV_4SJm25NwBp7FfRs,R_3fCPl2ygUlE6coc,4.158635e+08,73JP1U8,R,2026-08-01,CAK,ORD,3734,...,E7Q,Y,25.75,NaN,NaN,NaN,NaN,NaN,0,Detractor
2,2,SV_4SJm25NwBp7FfRs,R_24tIsphEEkOGZYp,3.656942e+08,0M99JX4,G,2026-07-19,AUS,PHL,1407,...,A19,Y,50.50,NaN,NaN,NaN,NaN,NaN,9,Promoter
3,3,SV_4SJm25NwBp7FfRs,R_705IoAtI9yevcsX,8.160689e+09,NaN,N,2026-02-01,LAX,MCO,1118,...,32E,Y,75.25,NaN,NaN,NaN,NaN,NaN,10,Promoter
4,4,SV_4SJm25NwBp7FfRs,R_7EnNVyqLDZekiOt,8.098126e+09,NaN,N,2025-10-14,EYW,CLT,3338,...,E7M,Y,1.00,NaN,NaN,NaN,NaN,NaN,0,Detractor


In [14]:
d2.SEG_DEP_DT.min(),d3.SEG_DEP_DT.min(), d1.SEG_DEP_DT.min()

('2026-01-01', '2024-01-01', '2024-01-01')